# {title}

状态：**{status}**。上传音频后按文件顺序分块推理；这不是实时麦克风测试。

[官方代码]({source_url}) · [模型权重]({model_url}) · [权重许可](https://github.com/netease-youdao/Confucius4-R2T2/blob/master/MODEL_LICENSE)。{license}。使用权重前请阅读许可。


## 1. 检查 GPU 和磁盘
在 Colab 中选择 GPU 运行时。官方测试 Python 3.12；Colab 环境若不兼容，请保留报错记录。


In [ ]:
import platform, shutil, subprocess, sys
from pathlib import Path
assert Path('/content').exists(), '请在 Colab 中运行'
assert shutil.which('nvidia-smi'), '请切换到 GPU 运行时'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
print('Python:', sys.version)
print('Platform:', platform.platform())
free_gib = shutil.disk_usage('/content').free / 2**30
print('空闲磁盘 GiB:', round(free_gib, 2))
assert free_gib >= 12, '至少需要 12 GiB 空闲磁盘用于权重和依赖'


## 2. 安装固定源码和依赖
官方源码固定到指定提交。依赖包含 vLLM，安装后若提示重启运行时，请重启并从头执行。


In [ ]:
SOURCE_REVISION = '72b81be7070095410b75ed2e11998e347f655218'
SOURCE_DIR = Path('/content/Confucius4-R2T2')
if not SOURCE_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/netease-youdao/Confucius4-R2T2.git', str(SOURCE_DIR)], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'fetch', 'origin', SOURCE_REVISION], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', SOURCE_REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE_DIR), 'huggingface_hub'], check=True)
print('source:', subprocess.check_output(['git', '-C', str(SOURCE_DIR), 'rev-parse', 'HEAD'], text=True).strip())


## 3. 下载固定版本权重并加载
权重只写入 `/content/models/confucius4-r2t2`。首次下载约 4.1 GB；记录权重文件的 SHA-256。


In [ ]:
import hashlib, importlib.metadata as metadata, os
os.environ['HF_HOME'] = '/content/models/.hf-cache'
from huggingface_hub import snapshot_download
import torch
MODEL_REVISION = 'd07b4a593610c16074cfc941fba052a7a12eeda3'
MODEL_DIR = Path('/content/models/confucius4-r2t2')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id='netease-youdao/Confucius4-R2T2', revision=MODEL_REVISION, local_dir=str(MODEL_DIR))
for name in ('model.safetensors', 'tokenizer.json'):
    path = MODEL_DIR / name
    if path.exists():
        digest = hashlib.file_digest(path.open('rb'), 'sha256').hexdigest()
        print(name, path.stat().st_size, digest)
for pkg in ('confucius4-r2t2', 'qwen-asr', 'vllm', 'torch', 'librosa', 'huggingface_hub'):
    print(pkg, metadata.version(pkg))
print('CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
from r2t2 import R2T2ASRModel
asr = R2T2ASRModel.LLM(model=str(MODEL_DIR), gpu_memory_utilization=0.4, max_model_len=16384, max_new_tokens=4)
print('loaded:', MODEL_DIR, MODEL_REVISION)


## 4. 上传一条音频
音频转成 16 kHz 单声道；默认语言为中文，可按输入修改为 `None` 自动检测。


In [ ]:
from google.colab import files
from IPython.display import Audio, display
import librosa
uploaded = files.upload()
assert len(uploaded) == 1, '请只上传一条音频'
audio_name, audio_bytes = next(iter(uploaded.items()))
audio_path = Path('/content') / Path(audio_name).name
audio_path.write_bytes(audio_bytes)
wav, sample_rate = librosa.load(str(audio_path), sr=16000, mono=True)
assert len(wav) > 0, '音频为空'
print('输入:', audio_path.name, '时长 s:', round(len(wav) / sample_rate, 2))
display(Audio(wav, rate=sample_rate))


## 5. 分块转写与记录
按官方 README 的 160 ms 分块接口处理已有文件。`stable_prefix` 是已确认前缀，最终结果来自 flush 后的 `state.text`。计时包含分块调用和 flush，不含下载、加载、上传和音频解码；单次耗时不代表实时延迟或稳定性能。


In [ ]:
import time
LANGUAGE = 'Chinese'
state = asr.init_streaming_state(context='', language=LANGUAGE, unfixed_chunk_num=0, unfixed_token_num=1, chunk_size_sec=0.16)
step = int(0.16 * sample_rate)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
updates = []
for pos in range(0, len(wav), step):
    _, stable_prefix = asr.streaming_transcribe(wav[pos:pos + step], state, max_new_tokens=4)
    if not updates or stable_prefix != updates[-1][1]:
        updates.append((round(min(pos + step, len(wav)) / sample_rate, 2), stable_prefix))
asr.finish_streaming_transcribe(state)
torch.cuda.synchronize()
elapsed_s = time.perf_counter() - start
print('稳定前缀更新（音频位置 s, 文本）:')
for update in updates:
    print(update)
print('最终转写:', state.text)
print('单次处理耗时 s:', round(elapsed_s, 3))
print('GPU 峰值 allocated MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1))
print('GPU 峰值 reserved MiB:', round(torch.cuda.max_memory_reserved() / 2**20, 1))
print('CER/WER: 待测（无参考文本）')


## 结果边界
仓库状态：**{status}**。该 Notebook 尚无 Colab GPU 成功日志；安装、加载与分块推理都需要实际验证。文件分块模拟不等于实时麦克风链路。
